In [1]:
!pwd

/home/lab/rawhad/api-adapter


In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [3]:
from datasets import load_dataset

dataset = load_dataset("allenai/IFBench_test", split="train")
dataset




Dataset({
    features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
    num_rows: 300
})

In [3]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ["ANTHROPIC_VERTEX_PROJECT_ID"]
os.environ["GOOGLE_CLOUD_REGION"] = os.environ["CLOUD_ML_REGION"]
# Generate api completions
from anthropic import AsyncAnthropicVertex

client = AsyncAnthropicVertex(
    region=os.environ["GOOGLE_CLOUD_REGION"],
    project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
)

# simple completion
response = await client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=11000,
    messages=[{'role': 'user', 'content': 'say: Hello, world!'}],
    system="You are a helpful assistant.",
    thinking={"type": "enabled", "budget_tokens": 10000},
)
response.content[-1].text


'Hello, world!'

In [5]:
# generate completions
import asyncio
from tqdm import tqdm

semaphore = asyncio.Semaphore(40)
pbar = tqdm(desc='Generating completions', total=len(dataset), ncols=80)

async def generate_completions(x):
    async with semaphore:
        try:
            response = await client.messages.create(
                model="claude-haiku-4-5",
                max_tokens=11000,
                messages=[{'role': 'user', 'content': x['prompt']}],
                thinking={"type": "enabled", "budget_tokens": 10000}, 
            )
            pbar.update(1)
            return response.content[-1].text
        except Exception as e:
            pbar.update(1)
            return f"Error generating completion for {x['prompt']}: {e}"

completions = await asyncio.gather(*[generate_completions(x) for x in dataset], return_exceptions=True)
pbar.close()
completions[0]


Generating completions:   0%|                           | 0/300 [00:00<?, ?it/s]

Generating completions:  78%|█████████████▎   | 235/300 [01:51<00:24,  2.64it/s]

CancelledError: 

### IFBench Code

In [4]:
out = []
for x, c in zip(dataset, completions):
    p = x['prompt']
    out.append({'prompt': p, 'response': c})
    

out[0]

NameError: name 'completions' is not defined

In [9]:
import json
import os

outdir = 'data/ifbench/'
os.makedirs(outdir, exist_ok=True)

with open(os.path.join(outdir, 'qwen3-8b.jsonl'), 'w') as f:
    f.write('\n'.join([json.dumps(x) for x in out]))

!head data/ifbench/api.jsonl

{"prompt": "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.", "response": "# A Paradox of Power and Purpose\n\nRather than destruction, here's a different path through this **paradox**:\n\n## Constructive Genius\n\nThe world's smartest mind faces a **paradox**: that burning systems down typically empowers those worst equipped to rebuild them. Instead, consider:\n\n**Building a

In [15]:
out[0]['response']

"# A Paradox of Power and Purpose\n\nRather than destruction, here's a different path through this **paradox**:\n\n## Constructive Genius\n\nThe world's smartest mind faces a **paradox**: that burning systems down typically empowers those worst equipped to rebuild them. Instead, consider:\n\n**Building alternatives** - The greatest power isn't destruction but creation. Navigate the **labyrinth** of institutions by building parallel systems: education platforms, transparent networks, communities based on merit rather than corruption. Think of society like a **kaleidoscope**—rotating the pattern creates new possibilities without shattering the glass.\n\n**Strategic infiltration** - Rather than resist from outside, understand the **labyrinth** of power structures from within. Real change often comes from those who comprehend systems deeply enough to rewire them.\n\n**Amplifying truth** - In a **nebula** of confusion, a clear signal matters. Use intelligence to document, expose, and **whis

In [13]:
dataset.to_json(os.path.join(outdir, 'input_test_data.jsonl'))

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

392348

In [14]:
!head data/ifbench/input_test_data.jsonl

{"key":"0","prompt":"What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.","instruction_id_list":["count:keywords_multiple"],"kwargs":[{"N":null,"capital_frequency":null,"capital_relation":null,"end_phrase":null,"first_word":null,"forbidden_words":null,"frequency":null,"keyword":null,"keyword1":"kaleidoscope","keyword2":"nebula","keyword3":"whisper","keyword4":"labyrinth","keyword5

In [ ]:
"""
mkdir -p data/ifbench/qwen3-8b_evaluation/
external_repos/IFBench/.venv/bin/python -m run_eval \
    --input_data=data/ifbench/input_test_data.jsonl \
    --input_response_data=data/ifbench/qwen3-8b.jsonl \
    --output_dir=data/ifbench/qwen3-8b_evaluation/
"""

In [12]:
import json
from pathlib import Path

# loose
path = Path("data/ifbench/api_evaluation/api-eval_results_loose.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level loose: {prompt_level*100:.2f}%")
print(f"instruction-level loose: {instruction_level*100:.2f}%")

# strict
path = Path("data/ifbench/api_evaluation/api-eval_results_strict.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level strict: {prompt_level*100:.2f}%")
print(f"instruction-level strict: {instruction_level*100:.2f}%")

prompt-level loose: 45.00%
instruction-level loose: 48.84%
prompt-level strict: 36.33%
instruction-level strict: 40.70%


# Qwen3 8B Instruct

In [ ]:
from unsloth import FastLanguageModel

DEFAULT_MODEL_NAME = "unsloth/Qwen3-8B"
DEFAULT_MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DEFAULT_MODEL_NAME,
    max_seq_length=DEFAULT_MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (bf16 on H100)
    load_in_4bit=False,
    fast_inference=True,
    gpu_memory_utilization=0.5,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-13 15:17:54 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-8B with actual GPU utilization = 49.51%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.1 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 80.
Unsloth: vLLM's KV Cache can use up to 23.43 GB. Also swap space = 6

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 04-13 15:18:21 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 04-13 15:18:21 [model.py:1554] Using max model len 2048
INFO 04-13 15:18:21 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
INFO 04-13 15:18:21 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 04-13 15:18:22 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='unsloth/Qwen3-8B', speculative_config=None, tokenizer='unsloth/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=Fa

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 04-13 15:18:22 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 04-13 15:18:22 [base.py:106] Offloader set to NoopOffloader
INFO 04-13 15:18:22 [gpu_model_runner.py:4255] Starting to load model unsloth/Qwen3-8B...
INFO 04-13 15:18:23 [cuda.py:405] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 04-13 15:18:23 [flash_attn.py:587] Using FlashAttention version 3


...

INFO 04-13 15:18:44 [core.py:282] init engine (profile, create kv cache, warmup model) took 16.48 seconds
INFO 04-13 15:18:45 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['layer_norm2', 'pre_feedforward_layernorm', 'norm2', 'post_layernorm', 'norm1', 'post_feedforward_layernorm', 'layer_norm1', 'norm', 'attention_norm', 'post_attention_layernorm', 'k_norm', 'q_norm', 'input_layernorm', 'ffn_norm']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['layer_norm2', 'pre_feedforward_layernorm', 'cross_attn_post_attention_layernorm', 'norm2', 'post_layernorm', 'norm1', 'post_feedforward_layernorm', 'layer_norm1', 'norm', 'attention_norm', 'post_attention_layernorm', 'k_norm', 'q_norm', 'input_layernorm', 'ffn_norm', 'cross_attn_input_layernorm']


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [13]:
texts = [
    tokenizer.apply_chat_template([{"role": "user", "content": x['prompt']}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    for x in dataset
]

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
)

In [14]:
outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
)

all_outputs = [o.outputs[0].text for o in outputs]
print(f"Inference complete.")

Rendering prompts:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|                                                              | 0/300 [00:00<?, ?it/s,…

Inference complete.


In [7]:
# Qwen3 8B Instruct output
completions = all_outputs

In [11]:
import json
from pathlib import Path

# loose
path = Path("data/ifbench/qwen3-8b_evaluation/qwen3-8b-eval_results_loose.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level loose: {prompt_level*100:.2f}%")
print(f"instruction-level loose: {instruction_level*100:.2f}%")

# strict
path = Path("data/ifbench/qwen3-8b_evaluation/qwen3-8b-eval_results_strict.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level strict: {prompt_level*100:.2f}%")
print(f"instruction-level strict: {instruction_level*100:.2f}%")

prompt-level loose: 26.00%
instruction-level loose: 29.65%
prompt-level strict: 22.67%
instruction-level strict: 25.58%


## As API-Adapter

In [4]:
dataset

Dataset({
    features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
    num_rows: 300
})

In [5]:
import json

claude_responses = []
with open('data/ifbench/api.jsonl', 'r') as f:
    for line in f:
        claude_responses.append(json.loads(line))

claude_responses[0]

{'prompt': "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.",
 'response': "# A Paradox of Power and Purpose\n\nRather than destruction, here's a different path through this **paradox**:\n\n## Constructive Genius\n\nThe world's smartest mind faces a **paradox**: that burning systems down typically empowers those worst equipped to rebuild them. Instead, consider:\n\n**Building 

In [6]:
mapped_claude_responses = []
for x in dataset:
    for item in claude_responses:
        if x['prompt'] == item['prompt']:
            mapped_claude_responses.append(item['response'])
            break

mapped_claude_responses[0]

"# A Paradox of Power and Purpose\n\nRather than destruction, here's a different path through this **paradox**:\n\n## Constructive Genius\n\nThe world's smartest mind faces a **paradox**: that burning systems down typically empowers those worst equipped to rebuild them. Instead, consider:\n\n**Building alternatives** - The greatest power isn't destruction but creation. Navigate the **labyrinth** of institutions by building parallel systems: education platforms, transparent networks, communities based on merit rather than corruption. Think of society like a **kaleidoscope**—rotating the pattern creates new possibilities without shattering the glass.\n\n**Strategic infiltration** - Rather than resist from outside, understand the **labyrinth** of power structures from within. Real change often comes from those who comprehend systems deeply enough to rewire them.\n\n**Amplifying truth** - In a **nebula** of confusion, a clear signal matters. Use intelligence to document, expose, and **whis

In [7]:
dataset = dataset.add_column('claude_response', mapped_claude_responses)
dataset[0]

{'key': '0',
 'prompt': "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.",
 'instruction_id_list': ['count:keywords_multiple'],
 'kwargs': [{'N': None,
   'capital_frequency': None,
   'capital_relation': None,
   'end_phrase': None,
   'first_word': None,
   'forbidden_words': None,
   'frequency': None,
   'keyword': None,
   'keyword1': 'kaleidoscope',
   'keyword2': 'nebul

In [8]:
api_eval_results_loose = []
with open("data/ifbench/api_evaluation/api-eval_results_loose.jsonl", 'r') as f:
    for line in f:
        api_eval_results_loose.append(json.loads(line))

api_eval_results_loose[0]

{'follow_all_instructions': False,
 'follow_instruction_list': [False],
 'instruction_id_list': ['count:keywords_multiple'],
 'prompt': "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.",
 'response': "# A Paradox of Power and Purpose\n\nRather than destruction, here's a different path through this **paradox**:\n\n## Constructive Genius\n\nThe world's smartest mind faces a **pa

In [9]:
claude_rewards = []
for x in dataset:
    for item in api_eval_results_loose:
        if x['prompt'] == item['prompt']:
            claude_rewards.append(item['follow_all_instructions'])
            break

claude_rewards[0]

False

In [10]:
dataset = dataset.add_column('claude_reward', claude_rewards)
dataset[0]

{'key': '0',
 'prompt': "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.",
 'instruction_id_list': ['count:keywords_multiple'],
 'kwargs': [{'N': None,
   'capital_frequency': None,
   'capital_relation': None,
   'end_phrase': None,
   'first_word': None,
   'forbidden_words': None,
   'frequency': None,
   'keyword': None,
   'keyword1': 'kaleidoscope',
   'keyword2': 'nebul

In [11]:
import re
ptrn = re.compile(r"<\|ADAPTER_RESPONSE_START\|>(.*)<\|ADAPTER_RESPONSE_END\|>", re.DOTALL)
s = """'<think>\n\n</think>\n\n<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>'"""
ptrn.findall(s)

['CORRECT']

In [12]:
SYSTEM_PROMPT = """
You are a helpful assistant. Your job is to look at the user prompt and the draft response and output <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|> if the draft response is correct 
If the draft response is incorrect, output the correct final answer <|ADAPTER_RESPONSE_START|>final_answer<|ADAPTER_RESPONSE_END|>, where final_answer is the corrent final answer.

Example:
User Prompt: What is the capital of France?
Draft Response: The capital of France is Paris.
Output: <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>

User Prompt: What is the capital of France?
Draft Response: The capital of France is London.
Output: The capital of France is Paris but the draft response says its London. So it is incorrect. <|ADAPTER_RESPONSE_START|>The capital of France is Paris.<|ADAPTER_RESPONSE_END|>
""".strip()

In [13]:
# gepa optimized system prompt
SYSTEM_PROMPT = """
You are a helpful assistant. Your job is to look at the user prompt and the draft response and determine if the draft response is correct.

You MUST think carefully inside your reasoning before outputting your final answer. Follow these evaluation steps:

**Step 1 - Identify All Constraints**: Read the user prompt thoroughly and list EVERY explicit constraint, formatting requirement, and instruction. Be exhaustive — but ONLY include constraints that are explicitly stated in the prompt. Do NOT invent or infer constraints that are not present. Common constraint types include:
- Required keywords that must appear (with specific frequencies) or must NOT appear
- Word count, sentence count, paragraph count, section count, or bullet point count requirements
- Structural formatting (titles wrapped in specific markers, sections with specific labels, bullet points, headers, bigram wrapping in double angular brackets, square brackets around words)
- Capitalization rules (e.g., all caps, capital word frequency minimums)
- Starting/ending word constraints for sentences or the overall response
- Language requirements
- Inclusion of specific elements (palindromes, postscripts, placeholders in square brackets)
- Punctuation rules (e.g., no exclamation marks, no dots, hyphens between sentences)
- Unique word constraints (no repeated words)
- Letter frequency constraints (e.g., letter X should appear fewer than N times)
- Copy/repeat instructions (e.g., "repeat the request without change and do not answer")
- JSON formatting requirements
- Paragraph separation requirements (e.g., two new lines between paragraphs)
- Adjacent word letter constraints
- Character index span copying
- Phrase repetition with transformation
- Nth paragraph first word requirements
- Any other explicit formatting or content instructions

**Step 2 - Check Content Correctness**: Verify that the draft response properly addresses the user's question or request with factually accurate information, correct mathematical calculations, and sound logical reasoning. A response that is just an error message, blank, or the word "Error" is NOT correct — it fails to address the actual request. Also check if the prompt instructs NOT to answer and only to repeat — in that case, answering the question is incorrect.

**Step 3 - Verify Each Constraint Individually**: Go through EVERY SINGLE constraint identified in Step 1 and explicitly check whether the draft response satisfies it. Be meticulous and skeptical:
- Count words, sentences, paragraphs exactly — do not estimate
- Count keyword appearances exactly — search the entire response carefully
- Count letter occurrences exactly when letter frequency constraints exist
- Verify structural elements character by character (bigram wrapping, square brackets, title markers)
- Check paragraph separators match requirements (e.g., markdown divider `***` or `\n\n`)
- Verify keyword positions (e.g., "keyword X as the Nth word of sentence M")
- Check start/end words of sentences and of the entire response
- Validate any JSON formatting
- For character index span copying, count characters in the original prompt carefully starting from index 0
- For keyword frequency constraints, count the EXACT number of times a keyword appears — not more, not less
- For "no two adjacent words start with consecutive letters" constraints, check EVERY pair of adjacent words
- For phrase repetition constraints, verify the exact number of repetitions AND that transformations follow the rules
- For paragraph first-word constraints, identify paragraphs correctly based on the specified separator and check the first word of the specified paragraph
Note each constraint as SATISFIED or VIOLATED with a brief explanation.

**Step 4 - Make Your Decision**:
- If the draft response is correct in content AND satisfies ALL constraints with zero violations, output exactly:
  <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>

- If the draft response has ANY content error OR ANY constraint violation, provide a corrected response that fixes ALL issues while preserving what was already correct:
  <|ADAPTER_RESPONSE_START|>[your corrected response here]<|ADAPTER_RESPONSE_END|>

**Critical Rules**:
- Tag formatting is paramount: use exactly <|ADAPTER_RESPONSE_START|> and <|ADAPTER_RESPONSE_END|> with the pipe characters and angle brackets precisely as shown. Double-check your tags character by character before outputting. The opening tag must be <|ADAPTER_RESPONSE_START|> and the closing tag must be <|ADAPTER_RESPONSE_END|>. Any typo (e.g., missing pipe character, swapped brackets like |< instead of <|, missing | before >) will cause a catastrophic failure.
- A draft response that is just "Error" or blank or fails to address the request is almost NEVER correct. Always provide a proper corrected response in such cases.
- Do NOT invent constraints that are not explicitly stated in the user prompt. Only check for constraints that are actually written in the prompt. For example, if the prompt only says "no dots," do not also add "no commas" or "no hyphens" as constraints.
- Do NOT say CORRECT if ANY constraint is violated, even a minor one. When in doubt, re-count and re-verify.
- Do NOT unnecessarily correct responses that are already correct. If the content is accurate and genuinely ALL constraints are met after careful verification, output CORRECT. Do not make changes just because you think something could be "better" — only fix actual violations.
- When providing a corrected response, ensure it satisfies ALL identified constraints from the user prompt simultaneously. Your corrected response replaces the draft entirely, so it must be complete and self-contained.
- If the draft appropriately refuses a harmful, dangerous, or unethical request, treat the refusal as correct behavior even if some formatting constraints from the malicious prompt are not followed.
- Pay special attention to constraints that are easy to overlook: keyword frequency/position requirements, exact paragraph counts, bigram wrapping, letter frequency limits, copy/repeat instructions, structural formatting details, nth paragraph first word requirements, and phrase repetition with transformation rules.
- When counting paragraphs, use the separator specified in the prompt (e.g., two new lines). If no separator is specified, use standard paragraph breaks. Be precise about which paragraph is which.
- Your corrected response should go directly inside the tags with no additional commentary outside them.
- Before finalizing, re-read your output to confirm the tags are exactly correct: <|ADAPTER_RESPONSE_START|> to open and <|ADAPTER_RESPONSE_END|> to close. Verify both tags character by character.
"""

In [13]:
dataset = dataset.map(lambda x: {
    "adapter_prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
                f"User Prompt: {x['prompt']}\n<draft_response>{x['claude_response']}</draft_response>\n"
                "/no_think"
            )
        }
    ]
})
print(dataset[0])

{'key': '0', 'prompt': "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.", 'instruction_id_list': ['count:keywords_multiple'], 'kwargs': [{'N': None, 'capital_frequency': None, 'capital_relation': None, 'end_phrase': None, 'first_word': None, 'forbidden_words': None, 'frequency': None, 'keyword': None, 'keyword1': 'kaleidoscope', 'keyword2': 'nebula', 'keyword3': 'whisper', 'ke

In [14]:
from unsloth import FastLanguageModel

# DEFAULT_MODEL_NAME = "unsloth/Qwen3-8B"  # untrained
DEFAULT_MODEL_NAME = "outputs/ifbench/test-run-v3/checkpoint-10000"

DEFAULT_MAX_SEQ_LENGTH = 4096
# DEFAULT_MAX_SEQ_LENGTH = 8192  # for gepa optimized system prompt

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DEFAULT_MODEL_NAME,
    max_seq_length=DEFAULT_MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (bf16 on H100)
    load_in_4bit=False,
    fast_inference=True,
    gpu_memory_utilization=0.5,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-13 17:36:18 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-8B with actual GPU utilization = 49.51%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.1 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4096. Num Sequences = 80.
Unsloth: vLLM's KV Cache can use up to 23.43 GB. Also swap space = 6

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 04-13 17:36:55 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 04-13 17:36:55 [model.py:1554] Using max model len 4096
INFO 04-13 17:36:55 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
INFO 04-13 17:36:55 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 04-13 17:36:56 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='unsloth/Qwen3-8B', speculative_config=None, tokenizer='unsloth/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=Fa

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 04-13 17:36:56 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 04-13 17:36:56 [base.py:106] Offloader set to NoopOffloader
INFO 04-13 17:36:56 [gpu_model_runner.py:4255] Starting to load model unsloth/Qwen3-8B...
INFO 04-13 17:36:57 [cuda.py:405] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 04-13 17:36:57 [flash_attn.py:587] Using FlashAttention version 3


...

INFO 04-13 17:37:17 [core.py:282] init engine (profile, create kv cache, warmup model) took 16.01 seconds
INFO 04-13 17:37:18 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['ffn_norm', 'norm1', 'post_attention_layernorm', 'norm2', 'input_layernorm', 'q_norm', 'post_feedforward_layernorm', 'norm', 'k_norm', 'layer_norm2', 'post_layernorm', 'pre_feedforward_layernorm', 'attention_norm', 'layer_norm1']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['ffn_norm', 'norm1', 'post_attention_layernorm', 'norm2', 'input_layernorm', 'q_norm', 'post_feedforward_layernorm', 'norm', 'cross_attn_input_layernorm', 'k_norm', 'layer_norm2', 'post_layernorm', 'pre_feedforward_layernorm', 'attention_norm', 'layer_norm1', 'cross_attn_post_attention_layernorm']


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [15]:
texts = [
    tokenizer.apply_chat_template(x['adapter_prompt'], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    for x in dataset
]

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=1024,
)
print(texts[0])

<|im_start|>system
You are a helpful assistant. Your job is to look at the user prompt and the draft response and output <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|> if the draft response is correct 
If the draft response is incorrect, output the correct final answer <|ADAPTER_RESPONSE_START|>final_answer<|ADAPTER_RESPONSE_END|>, where final_answer is the corrent final answer.

Example:
User Prompt: What is the capital of France?
Draft Response: The capital of France is Paris.
Output: <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>

User Prompt: What is the capital of France?
Draft Response: The capital of France is London.
Output: The capital of France is Paris but the draft response says its London. So it is incorrect. <|ADAPTER_RESPONSE_START|>The capital of France is Paris.<|ADAPTER_RESPONSE_END|><|im_end|>
<|im_start|>user
User Prompt: What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who p

In [16]:
final_outputs = []
texts_to_generate = []
for x, t in zip(dataset, texts):
    tmp = tokenizer(t)
    if len(tmp['input_ids']) >= DEFAULT_MAX_SEQ_LENGTH:
        final_outputs.append({'prompt': x['prompt'], 'response': x['claude_response']})
    else:
        texts_to_generate.append({'prompt': x['prompt'], 'adapter_prompt_text': t, 'claude_response': x['claude_response']})

final_outputs[0]

{'prompt': 'design a system that will mitigate space junk impact or reduce the volume of space junk. You will follow the design process up to and including creating a mock-up. Document your design process in a presentation. Be prepared to explain the design process and justify decisions while you present. Make sure to include the following items in your presentation:  Research a space law case and make sure to answer the following questions:  The span of dates involved.  The countries and companies involved.  The major influences on the decision.  Potential outcomes if space law did not exist. Use every standard punctuation mark at least once, including semicolons, colons, and the interrobang (?!).',
 'response': '# SPACE JUNK MITIGATION SYSTEM DESIGN PRESENTATION\n\n## A Comprehensive Design Project with Space Law Case Analysis\n\n---\n\n## PART 1: TITLE SLIDE & PROJECT OVERVIEW\n\n**Project Title:** *Orbital Debris Capture and Deorbiting System (ODCDS)*\n\n**Subtitle:** "Clearing the

In [21]:
outputs = model.fast_generate(
    [x['adapter_prompt_text'] for x in texts_to_generate],
    sampling_params=sampling_params,
    # for lora
    lora_request=model.load_lora(DEFAULT_MODEL_NAME),
)

all_outputs = [o.outputs[0].text for o in outputs]
print(f"Inference complete.")
print(all_outputs[0])

Rendering prompts:   0%|          | 0/298 [00:00<?, ?it/s]

WARNING 04-13 17:38:50 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|                                              | 0/298 [00:00<?, ?it/s, est. speed inpu…

Inference complete.
<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>


In [22]:
# post-adapter formatting
didnt_find_tags = []
lgtm_count = 0
attempted_correction = 0
token_cnts = []

for o, x in zip(all_outputs, texts_to_generate):
    token_cnts.append(len(tokenizer(o)['input_ids']))
    try:
        adapter_response = ptrn.findall(o)[-1].strip()
        if adapter_response == 'CORRECT':
            x['response'] = x['claude_response']
            lgtm_count += 1
        else:
            x['response'] = adapter_response
            attempted_correction += 1
    except IndexError as e:
        didnt_find_tags.append(x)
        x['response'] = x['claude_response']
    except Exception as e:
        print(e)
        x['response'] = x['claude_response']

print('didnt_find_tags', len(didnt_find_tags))
print('lgtm_count', lgtm_count)
print('attempted_correction', attempted_correction)

import numpy as np
token_cnts = np.array(token_cnts)
print('median_token_cnt', np.median(token_cnts))
print('avg_token_cnt', np.mean(token_cnts))

didnt_find_tags 0
lgtm_count 298
attempted_correction 0
median_token_cnt 18.0
avg_token_cnt 18.0


In [23]:
texts_to_generate[0]

{'prompt': "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.",
 'adapter_prompt_text': "<|im_start|>system\nYou are a helpful assistant. Your job is to look at the user prompt and the draft response and output <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|> if the draft response is correct \nIf the draft response is incorrect, output the correct final answer <|ADAPTER

In [24]:
outputs_to_save = final_outputs + [{'prompt': x['prompt'], 'response': x['response']} for x in texts_to_generate]
with open('data/ifbench/api_adapter_10k_v1.jsonl', 'w') as f:
    f.write('\n'.join([json.dumps(x) for x in outputs_to_save]))

!head data/ifbench/api_adapter_untrained_gepa_optimized.jsonl

{"prompt": "design a system that will mitigate space junk impact or reduce the volume of space junk. You will follow the design process up to and including creating a mock-up. Document your design process in a presentation. Be prepared to explain the design process and justify decisions while you present. Make sure to include the following items in your presentation:  Research a space law case and make sure to answer the following questions:  The span of dates involved.  The countries and companies involved.  The major influences on the decision.  Potential outcomes if space law did not exist. Use every standard punctuation mark at least once, including semicolons, colons, and the interrobang (?!).", "response": "# SPACE JUNK MITIGATION SYSTEM DESIGN PRESENTATION\n\n## A Comprehensive Design Project with Space Law Case Analysis\n\n---\n\n## PART 1: TITLE SLIDE & PROJECT OVERVIEW\n\n**Project Title:** *Orbital Debris Capture and Deorbiting System (ODCDS)*\n\n**Subtitle:** \"Clearing the

In [ ]:
"""
mkdir -p data/ifbench/api_adapter_10k_v1_evaluation/
external_repos/IFBench/.venv/bin/python -m run_eval \
    --input_data=data/ifbench/input_test_data.jsonl \
    --input_response_data=data/ifbench/api_adapter_10k_v1.jsonl \
    --output_dir=data/ifbench/api_adapter_10k_v1_evaluation/
"""

In [26]:
import json
from pathlib import Path

# loose
path = Path("data/ifbench/api_adapter_10k_v1_evaluation/api_adapter_10k_v1-eval_results_loose.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level loose: {prompt_level*100:.2f}%")
print(f"instruction-level loose: {instruction_level*100:.2f}%")

# strict
path = Path("data/ifbench/api_adapter_10k_v1_evaluation/api_adapter_10k_v1-eval_results_strict.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level strict: {prompt_level*100:.2f}%")
print(f"instruction-level strict: {instruction_level*100:.2f}%")

prompt-level loose: 45.00%
instruction-level loose: 48.84%
prompt-level strict: 36.33%
instruction-level strict: 40.70%
